# 🫀 CV Intel RAG — Notebook 1: Ingest & Index

**Run this once.** It fetches CV/DM/CKD data from 7 sources, embeds with
BGE-M3 on the free T4 GPU, and saves the index to your Google Drive so
Notebook 2 can start demos in seconds.

**Time:** ~10 min total (3 min fetch + 7 min embed)

### Pipeline

```
PubMed ───┐
CT.gov ───┤
openFDA ──┼──► SQLite ──► chunker ──► BGE-M3 (T4) ──► ChromaDB ──► Drive
4 RSS ────┘                                                          │
                                                                 Notebook 2
```

### Shared Drive layout

```
MyDrive/cv-intel-rag/
├── data/
│   ├── cv_intel.db        ← SQLite records
│   ├── chroma/            ← ChromaDB persistent files
│   └── cv-intel-rag-data.tar.gz  ← to upload to HF Space
└── logs/


In [ ]:
%pip install -q \
  fastapi uvicorn pydantic pydantic-settings sqlalchemy \
  httpx beautifulsoup4 feedparser openai \
  chromadb sentence-transformers rank-bm25 tiktoken \
  opentelemetry-api==1.38.0 \
  opentelemetry-sdk==1.38.0 \
  opentelemetry-proto==1.38.0 \
  opentelemetry-exporter-otlp-proto-common==1.38.0 \
  opentelemetry-exporter-otlp-proto-http==1.38.0


## Step 2 — Mount Google Drive



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/cv-intel-rag')
(PROJECT_DIR / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'data' / 'raw').mkdir(exist_ok=True)
(PROJECT_DIR / 'logs').mkdir(exist_ok=True)


## Step 3 — Typhoon API key (Colab Secrets)

1. Go to left sidebar → 🔑 **Secrets** → **+ Add new secret**
2. Name: `TYPHOON_API_KEY` · Value: your key from
   [playground.opentyphoon.ai/settings/api-key](https://playground.opentyphoon.ai/settings/api-key)


In [ ]:
import os
from google.colab import userdata

try:
    os.environ['TYPHOON_API_KEY'] = userdata.get('TYPHOON_API_KEY')
    print('✓ Typhoon key loaded (len=%d)' % len(os.environ['TYPHOON_API_KEY']))
except Exception as e:
    print(f'⚠️  TYPHOON_API_KEY not found in Colab Secrets: {e}')
    print('   Add it via the 🔑 Secrets panel (left sidebar), then re-run this cell.')
    print('   LLM calls will fail until the key is set.')
    os.environ.setdefault('TYPHOON_API_KEY', '')


## Step 4 — Clone the repo



In [ ]:
# ⚠️  Set your GitHub username here before running this cell
GITHUB_USERNAME = 'siriponsri'

import subprocess
subprocess.run(['rm', '-rf', '/content/cv-intel-rag'], check=False)
%cd /content
!git clone -q https://github.com/{GITHUB_USERNAME}/cv-intel-rag.git
%cd /content/cv-intel-rag

# Install the package in editable mode.
# This registers `src` and all its sub-packages (src.config, src.connectors,
# src.models, src.rag, ...) as a proper installed package, overriding any
# conflicting `src` namespace that Colab's Python 3.12 may have from
# site-packages (e.g. chromadb / opentelemetry deps).
%pip install -e . -q --no-deps
print('✓ package installed in editable mode')


## Step 5 — Configure environment



In [ ]:
import os

DB_PATH     = PROJECT_DIR / 'data' / 'cv_intel.db'
CHROMA_PATH = PROJECT_DIR / 'data' / 'chroma'

# Point all subprocess calls at the repo root so `import src` always resolves correctly.
os.environ['PYTHONPATH'] = '/content/cv-intel-rag'

env = f'''APP_ENV=development
DATABASE_URL=sqlite:///{DB_PATH}
CHROMA_PATH={CHROMA_PATH}
DEFAULT_LLM_PROVIDER=typhoon_api
TYPHOON_BASE_URL=https://api.opentyphoon.ai/v1
TYPHOON_MODEL=typhoon-v2.5-30b-a3b-instruct
EMBED_MODEL_NAME=BAAI/bge-m3
EMBED_DEVICE=cuda
EMBED_BATCH_SIZE=32
'''
open('.env', 'w').write(env)
print('✓ .env written')


## Step 6 — Initialize DB + run ingestion

Fetches ~30 records from each of 7 connectors (~200 records total).


In [ ]:
import sys, os, time, subprocess

LIMIT_PER_SOURCE = 30
REPO   = '/content/cv-intel-rag'
PYTHON = sys.executable   # same interpreter as the notebook kernel
_env   = {**os.environ, 'PYTHONPATH': REPO}

# init DB
result = subprocess.run([PYTHON, 'scripts/init_db.py'],
                        cwd=REPO, env=_env,
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR (init_db):', result.stderr[-2000:])

# run ingestion
t0 = time.time()
result = subprocess.run(
    [PYTHON, 'scripts/run_ingestion.py', '--limit', str(LIMIT_PER_SOURCE)],
    capture_output=True, text=True,
    cwd=REPO,
    env=_env,
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print('STDERR (ingestion):', result.stderr[-2000:])
print(f'⏱ {time.time()-t0:.0f}s')


### 📊 Ingestion summary chart



In [ ]:
import sqlite3, matplotlib.pyplot as plt

conn = sqlite3.connect(DB_PATH)
rows = conn.execute('SELECT source_name, COUNT(*) FROM records GROUP BY source_name ORDER BY 2 DESC').fetchall()
conn.close()

sources = [r[0] for r in rows]
counts  = [r[1] for r in rows]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(sources, counts, color='#c0392b')
ax.set_xlabel('records')
ax.set_title(f'Ingested records by source  (total={sum(counts)})')
for b, c in zip(bars, counts):
    ax.text(b.get_width()+0.3, b.get_y()+b.get_height()/2, str(c), va='center')


## Step 7 — Build the vector index with BGE-M3

Chunks each record's raw_text, embeds with BGE-M3 on the T4 GPU, stores


In [ ]:
import time, sys, os, subprocess

REPO   = '/content/cv-intel-rag'
PYTHON = sys.executable
_env   = {**os.environ, 'PYTHONPATH': REPO}

t0 = time.time()
result = subprocess.run([PYTHON, 'scripts/rebuild_index.py'],
                        cwd=REPO, env=_env,
                        capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print('STDERR (rebuild_index):', result.stderr[-2000:])
print(f'⏱ {time.time()-t0:.0f}s')


In [ ]:
import chromadb
client = chromadb.PersistentClient(path=str(CHROMA_PATH))
coll = client.get_or_create_collection(name='cv_intel_chunks')
print(f'✓ ChromaDB contains {coll.count()} chunks')

# category distribution
import sqlite3
conn = sqlite3.connect(DB_PATH)
cat_rows = conn.execute('SELECT category, COUNT(*) FROM records GROUP BY category').fetchall()
conn.close()

fig, ax = plt.subplots(figsize=(7, 4))
cats = [r[0] for r in cat_rows]
nums = [r[1] for r in cat_rows]
ax.pie(nums, labels=cats, autopct='%1.0f%%', colors=plt.cm.Reds_r(range(40, 40+len(cats)*30, 30)))


## Step 8 — Sanity check: one retrieval query



In [ ]:
from src.rag.retriever import HybridRetriever

retriever = HybridRetriever(top_k=5)
query = 'SGLT2 inhibitors for heart failure with preserved ejection fraction'
hits = retriever.retrieve(query)

print(f'Query: {query}\n')
for i, h in enumerate(hits, 1):
    print(f'[{i}] score={h.score:.3f}  source={h.metadata.get("source_name")}')
    print(f'    title: {h.metadata.get("title","")[:100]}')
    print(f'    text:  {h.text[:160]}...')


## Step 9 — Package data/ for Hugging Face Space upload

Creates a tar.gz archive of the index. You'll commit this to your HF Space


In [ ]:
import tarfile

TAR_PATH = PROJECT_DIR / 'data' / 'cv-intel-rag-data.tar.gz'
with tarfile.open(TAR_PATH, 'w:gz') as tar:
    tar.add(DB_PATH, arcname='data/cv_intel.db')
    tar.add(CHROMA_PATH, arcname='data/chroma')

size_mb = TAR_PATH.stat().st_size / 1024 / 1024
print(f'✓ archive: {TAR_PATH}  ({size_mb:.1f} MB)')


## 🎉 Done — now open `02_demo_visualization.ipynb`

The demo notebook loads from this same Drive folder and shows the full
